***** UNIT 3 CODE *****

The goal of this code is to expand the google review analysis by extracting all named entities from the feedback, including but not limited to product or service names, specific to the selected business, mentions of competitors, geographic places, etc.

In [45]:
path = '/content/drive/My Drive/CIS617_TextAnalytics/TextAnalyticsCode_FordKCAP'

import pandas as pd
reviews_df = pd.read_csv(path+'/reviews.csv')
reviews_df.head(3)

,review_position,link,rating,date,iso_date,iso_date_of_last_edit,source,review_id,user,snippet,extracted_snippet,likes,images
0,1,https://www.google.com/maps/reviews/data=!4m8!...,5.0,5 months ago,2025-10-29T03:17:23Z,2025-10-29T03:17:23Z,Google,Ci9DQUlRQUNvZENodHljRjlvT214UlZHNXlhbXBUWjI1Rl...,"{'name': 'Sarah Tasciotti', 'link': 'https://w...",Easy drop and hook. Never had a problem with s...,"{'original': ""Easy drop and hook. Never had a ...",0,NaN
1,2,https://www.google.com/maps/reviews/data=!4m8!...,1.0,3 months ago,2025-12-28T14:33:40Z,2025-12-28T14:33:40Z,Google,Ci9DQUlRQUNvZENodHljRjlvT25rMFMxRjBSbWxWY213ek...,"{'name': 'Kaden Ash Richardson', 'link': 'http...",You make living across from you hell. Constant...,{'original': 'You make living across from you ...,0,NaN
2,3,https://www.google.com/maps/reviews/data=!4m8!...,1.0,Edited 5 months ago,2025-10-02T11:39:10Z,2025-10-10T14:14:55Z,Google,Ci9DQUlRQUNvZENodHljRjlvT2pCaGFGUlBNMnR1UXpsdV...,"{'name': 'T Gold', 'link': 'https://www.google...",Trucker beware... Mostly everyone is helpful e...,"{'original': ""Trucker beware... Mostly everyon...",0,NaN


In [46]:
import pandas as pd
import spacy
import requests
from bs4 import BeautifulSoup
nlp = spacy.load("en_core_web_sm")

In [47]:
# Store ['snippet'] column as a single line string containing all user-submitted reviews
reviewsString = reviews_df['snippet'].to_string(index=False)
reviewsString = reviewsString.replace("\n", " ")

Create a new dataset that describes all entities, their categories, and their appearances in data. 1 of 5.

In [48]:
# Displays the names of the entities,
# their start and end positions in the text,
# and their predicted labels.
doc = nlp(reviewsString)

for ent in doc.ents:
    print(ent.text, ent.start_char, ent.end_char, ent.label_)

Constantl 89 98 PERSON
night 158 163 TIME
Ford 328 332 ORG
6 339 340 CARDINAL
24 hours 381 389 TIME
2018 462 466 DATE
12 hours 594 602 TIME
Two 765 768 CARDINAL
this morning 836 848 TIME
Saturday 923 931 DATE
Sundays 1005 1012 DATE
Ford 1186 1190 ORG
Trailer 1343 1350 ORG
Feb 22 1357 1363 DATE
2022 1365 1369 DATE
750 1574 1577 CARDINAL
2 1742 1743 CARDINAL
Ford 1840 1844 ORG
Claycomo 1868 1876 PERSON
Caucasian 1901 1910 NORP
Henry Ford 2040 2050 PERSON
pioneer 2063 2070 ORG
7 days 2091 2097 DATE
Bulk 2362 2366 PERSON
21/2 hours 2511 2521 TIME
Ford Asse 2741 2750 ORG
BASF 2786 2790 ORG
Drop 2805 2809 PERSON
Monday 2856 2862 DATE
4:12PM 2874 2880 TIME
1st 2920 2923 DATE
100% 2931 2935 PERCENT
first 2998 3003 ORDINAL
5 3023 3024 CARDINAL
12 3054 3056 CARDINAL
Frank Rogers 3181 3193 PERSON
5 3293 3294 CARDINAL
130 pm 3377 3383 TIME
2 3390 3391 CARDINAL
Garison Rd 3430 3440 FAC
Trailer 3447 3454 PERSON
unl 3512 3515 PERSON
LOADS 3561 3566 PERSON
three 3685 3690 CARDINAL
Ford 3691 3695 ORG
2

Create a new dataset that describes all entities, their categories, and their appearances in data. 2 of 5.

In [49]:
# Visualize named entities in the processed doc object
# by highlighting them in the text with their respective
# categories such as person, organization, location etc.
from spacy import displacy
displacy.render(doc, style="ent")

Create a new dataset that describes all entities, their categories, and their appearances in data. 3 of 5.

In [50]:
# Create a list of tuples where each tuple contains the
# text, label (type) and lemma (base form) of each named
# entity found in the processed doc object.

entities = [(ent.text, ent.label_, ent.lemma_) for ent in doc.ents]
entities_df = pd.DataFrame(entities, columns=['text', 'type', 'lemma'])

entities_df.head(5)

,text,type,lemma
0,Constantl,PERSON,Constantl
1,night,TIME,night
2,Ford,ORG,Ford
3,6,CARDINAL,6
4,24 hours,TIME,24 hour


Create a new dataset that describes all entities, their categories, and their appearances in data. 4 of 5.

In [51]:
# Add new column that contains the count of how many times each
# string in the 'text' column appears within the DataFrame.

entities_df['text'].value_counts()

entities_df['occurrences'] = entities_df['text'].map(entities_df['text'].value_counts())

entities_df.head(5)

,text,type,lemma,occurrences
0,Constantl,PERSON,Constantl,1
1,night,TIME,night,2
2,Ford,ORG,Ford,10
3,6,CARDINAL,6,1
4,24 hours,TIME,24 hour,1


Create a new dataset that describes all entities, their categories, and their appearances in data. 5 of 5.

In [52]:
# Add a column to entities_df that contains the original review locations by
# iterating through each review left by users 'snippets'.

entity_to_positions = {}

# Loop through the reviews
for index, row in reviews_df.iterrows():
    snippet = row['snippet']
    position = row['review_position'] # Assuming 'position' is 1-indexed as seen in head(3)

    if pd.notna(snippet): # Process only if the snippet is not NaN
        doc_snippet = nlp(str(snippet)) # Ensure snippet is a string
        for ent in doc_snippet.ents:
            entity_text = ent.text
            if entity_text not in entity_to_positions:
                entity_to_positions[entity_text] = []
            entity_to_positions[entity_text].append(position)

# Ensure lists contain unique positions for each entity text and are sorted
for entity_text in entity_to_positions:
    entity_to_positions[entity_text] = sorted(list(set(entity_to_positions[entity_text])))

# Map these collected positions to the existing entities_df
# If an entity_df['text'] is not found in the collected_positions, it will get an empty list
entities_df['review_positions'] = entities_df['text'].apply(lambda text: entity_to_positions.get(text, []))

entities_df.head(5)

,text,type,lemma,occurrences,review_positions
0,Constantl,PERSON,Constantl,1,[]
1,night,TIME,night,2,[4]
2,Ford,ORG,Ford,10,"[4, 7, 13, 21, 24, 33, 37, 40, 48, 57, 63, 69,..."
3,6,CARDINAL,6,1,[7]
4,24 hours,TIME,24 hour,1,"[8, 18]"


"Make appropriate changes to the dataset, e.g., create a list of words to exclude, and then write code to exclude those entities from the dataset."

In [53]:
# Remove entity duplicates.
entities_df = entities_df.drop_duplicates(subset=['text'])

# Filter out entities with type 'CARDINAL' due to the random use of numbers being labeled.
entities_df = entities_df[entities_df['type'] != 'CARDINAL']

# Sort entity occurrences in descending order for better understanding.
entities_df = entities_df.sort_values(by='occurrences', ascending=False)

entities_df.head(5)

,text,type,lemma,occurrences,review_positions
2,Ford,ORG,Ford,10,"[4, 7, 13, 21, 24, 33, 37, 40, 48, 57, 63, 69,..."
27,Drop,PERSON,drop,3,[56]
5,2018,DATE,2018,3,"[10, 114, 194]"
12,Trailer,ORG,trailer,2,"[27, 34, 68]"
1,night,TIME,night,2,[4]


Save the new updated dataset into a file named name_entities.csv

In [54]:
# Save entities to csv file in local Google Drive.
entities_df.to_csv(path+'/name_entities.csv', index=False)

Within the Colab Notebook, conduct descriptive analysis of the name entities in the dataset.

In [55]:
# Determine the total unique entities, total mentions across reviews,
# and number of distinct entity types

total_unique_entities = len(entities_df)
total_mentions = entities_df['occurrences'].sum()
distinct_entity_types = entities_df['type'].nunique()

print(f"Total unique entities: {total_unique_entities}")
print(f"Total mentions across reviews: {total_mentions}")
print(f"Distinct entity types: {distinct_entity_types}")

Total unique entities: 65
Total mentions across reviews: 82
Distinct entity types: 13


The named entity dataset consists of 65 unique entities with a total of 82 mentions across all reviews, spanning 13 distinct entity types. The relatively low total number of mentions compared to unique entities suggests that reviewer language is varied, with limited repetition of the same named entities. This diversity reflects a broad range of references made by reviewers rather than a narrow focus on a small set of recurring terms.

In [56]:
# Top 10 most frequent named entities
top_entities = entities_df.sort_values(by="occurrences", ascending=False).head(10)
top_entities

,text,type,lemma,occurrences,review_positions
2,Ford,ORG,Ford,10,"[4, 7, 13, 21, 24, 33, 37, 40, 48, 57, 63, 69,..."
27,Drop,PERSON,drop,3,[56]
5,2018,DATE,2018,3,"[10, 114, 194]"
12,Trailer,ORG,trailer,2,"[27, 34, 68]"
1,night,TIME,night,2,[4]
80,5 hours,TIME,5 hour,2,"[91, 100, 156, 191]"
56,F-150,PERSON,f-150,2,"[98, 149]"
0,Constantl,PERSON,Constantl,1,[]
6,12 hours,TIME,12 hour,1,[12]
10,Sundays,DATE,sunday,1,[20]


This assessment of the most frequently mentioned entities shows that “Ford” is the dominant entity, which is expected given that all reviews are centered on the Ford Kansas City Assembly Plant. Reviewers also frequently mention logistical elements and durations of time. The prominence of time-related entities indicates that wait times and length of stay are important aspects of the customer experience being described.

In [57]:
# Total occurrences by entity type
type_occurrences = entities_df.groupby("type")["occurrences"].sum().sort_values(ascending=False)
type_occurrences

,occurrences
type,
ORG,20
PERSON,20
DATE,17
TIME,15
WORK_OF_ART,2
MONEY,1
LAW,1
GPE,1
FAC,1


The distribution of entity types by total occurrences further reinforces these insights. The most frequent categories are ORG and PERSON, each with 20 occurrences, followed by DATE (17) and TIME (15). All other categories, such as WORK_OF_ART, MONEY, LAW, and GPE, appear only once or twice. This concentration suggests that reviewers primarily focus on organizations, individuals, and temporal aspects. There is a noteworthy presence of DATE and TIME entities which may highlight the importance of scheduling, delays, and operational timing. This may ultimately shape user/customer perceptions.


In [58]:
# Show how many unique entity rows belong to each category.

# Count of unique entities per type
type_counts = entities_df["type"].value_counts()
type_counts

,count
type,
PERSON,17
DATE,15
TIME,13
ORG,10
WORK_OF_ART,2
NORP,1
PERCENT,1
FAC,1
ORDINAL,1


When examining the number of unique entities within each category, a similar pattern emerges. ORG and PERSON again account for the largest share of unique entities, indicating that reviewers refer to a variety of organizations and individuals. DATE and TIME also show a relatively high number of unique entries, reflecting diverse references to specific days, years, and durations. In contrast, the remaining categories contain very few unique entities, suggesting that they play only a minor role in the overall dataset. This distribution suggests that the primary scope of discussion in the reviews is likely focused on the organization (Ford KCAP), human interactions, and time-related experiences.

In [59]:
# Identify the most common ORG, PERSON, DATE, and TIME entities.

# Function to display top entities for a selected type
def top_entities_by_type(entity_type, top_n=10):
    subset = entities_df[entities_df["type"] == entity_type].sort_values(by="occurrences", ascending=False).head(top_n)
    return subset[["text", "type", "occurrences"]]

# Print top Entities by type
print("Top ORG entities")
print(top_entities_by_type("ORG"))

print("\nTop PERSON entities")
print(top_entities_by_type("PERSON"))

print("\nTop DATE entities")
print(top_entities_by_type("DATE"))

print("\nTop TIME entities")
print(top_entities_by_type("TIME"))

Top ORG entities
              text type  occurrences
2             Ford  ORG           10
12         Trailer  ORG            2
25       Ford Asse  ORG            1
26            BASF  ORG            1
21         pioneer  ORG            1
53             CMV  ORG            1
55  KS CY Assembly  ORG            1
66          Subaru  ORG            1
69             CDL  ORG            1
93             HOT  ORG            1

Top PERSON entities
            text    type  occurrences
27          Drop  PERSON            3
56         F-150  PERSON            2
0      Constantl  PERSON            1
18      Claycomo  PERSON            1
20    Henry Ford  PERSON            1
23          Bulk  PERSON            1
35  Frank Rogers  PERSON            1
41           unl  PERSON            1
42         LOADS  PERSON            1
51       asbesto  PERSON            1

Top DATE entities
        text  type  occurrences
5       2018  DATE            3
10   Sundays  DATE            1
13    Feb 22  DATE    

Within the ORG category, “Ford” clearly dominates, emphasizing the central role of the company in all feedback. In the PERSON category, multiple individual references appear, though none are as dominant as “Ford,”. This likely indicates dispersed mentions of people rather than a focus on specific individuals. For DATE entities, values such as “2018” are among the most frequent, suggesting references to vehicle model years or specific time periods. Within the TIME category, expressions like “5 hours” and “12 hours” are especially prominent, reinforcing the importance of wait times and duration-related experiences.